# Capstone — Exploratory Snapshot Experiment (Temporal Validation Pending)

> **STATUS: NOT VALIDATED.** The available 30,000-row snapshot contains 90-day aggregates that overlap the recent 30-day period used to derive the decline target. A client-grouped split does not resolve this temporal leakage. All previously published Precision@K and ROC-AUC claims are withdrawn. This notebook is retained as a historical exploratory artifact; do not present its metrics as prospective model performance.
>
> A valid rebuild requires dated daily records, features strictly before a prediction cutoff, and a label window strictly after that cutoff. See [the remediation plan](../../docs/temporal-validation-remediation.md).

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sohila-Khaled-Abbas/flyrank-ml-internship-starter/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

**Author:** Sohila Khaled  
**Dataset:** 30,000-row anonymized snapshot (exploratory only)  
**Lane:** Refresh / Content Opportunity Scoring

## 1. Executive Summary — Historical Experiment Only

This notebook records the original snapshot-based experiment for content-refresh prioritization. Its feature and target windows overlap, so the results cannot support claims about future prediction or generalization to future periods.

The model-building cells below are preserved for reproducibility and audit. **Do not cite their metrics.**

In [1]:
# Capstone Environment & Dataset Setup
import pandas as pd
import numpy as np
import json
import os
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, precision_score

csv_path = "../../data/raw/content_refresh_anonymized.csv"
if not os.path.exists(csv_path):
    csv_path = "data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(csv_path)
print(f"Capstone Dataset Loaded: {len(df):,} rows × {df.shape[1]} columns across {df['client_id'].nunique()} clients.")


Capstone Dataset Loaded: 30,000 rows × 44 columns across 32 clients.


## 2. Problem Statement & Decision Framework

- **What decision does this improve?** Which decaying or underperforming content items should editorial teams prioritize for content refresh and SEO optimization first?
- **Who acts on the output?** Content editors and SEO strategists.
- **Cost of Errors:**
  - *False Positive (recommending a healthy page):* ~4.0 editorial hours wasted per unnecessary rewrite.
  - *False Negative (missing a high-impact page):* ~25%+ unrecovered organic traffic loss on high-volume URLs.

In [2]:
# Problem Scope Summary
down_count = (df["trend_direction"] == "down").sum()
high_vol_down = ((df["impressions_90d"] > df["impressions_90d"].median()) & (df["trend_direction"] == "down")).sum()
print(f"Decaying Pages in Scope: {down_count:,} ({down_count/len(df):.1%})")
print(f"High-Impact Decaying Pages (Impressions > Median): {high_vol_down:,} URLs")


Decaying Pages in Scope: 16,262 (54.2%)
High-Impact Decaying Pages (Impressions > Median): 8,906 URLs


## 3. Data Contract & Signal Audit — Snapshot Limitation

The original target proxy is derived from recent-period trend fields. The bundled snapshot also contains 90-day activity aggregates that include the same recent period. Therefore, removing explicit target columns does not make the remaining features temporally safe.

This historical audit is not a point-in-time leakage proof. See [temporal validation remediation](../../docs/temporal-validation-remediation.md).

In [3]:
# Data Contract Verification
assert df["content_id"].nunique() == len(df), "Grain check failed!"

# Handle avg_position=0 missing rank marker
df["pos_clean"] = np.where(df["avg_position"] == 0, 50.0, df["avg_position"])
df["traffic_drop"] = np.maximum(0, df["clicks_prev_30d"] - df["clicks_last_30d"])
df["target_binary"] = (df["traffic_drop"] > df["traffic_drop"].median()).astype(int)

print("Data Contract Verification: Grain unique (1 row = 1 content_id).")
print(f"Target binary decay prevalence (base rate): {df['target_binary'].mean():.1%}")


Data Contract Verification: Grain unique (1 row = 1 content_id).
Target binary decay prevalence (base rate): 22.7%


## 4. Historical Client-Grouped Split — Not Temporally Validated

The following code demonstrates the original client-grouped split only. It isolates clients, but the snapshot's feature windows still overlap the target window. The resulting model scores are exploratory and must not be reported as validated performance.

In [4]:
# Grouped Client Train/Test Split
clients = df["client_id"].unique()
np.random.seed(42)
train_clients = np.random.choice(clients, size=int(len(clients)*0.8), replace=False)

train_df = df[df["client_id"].isin(train_clients)].copy()
test_df = df[~df["client_id"].isin(train_clients)].copy()

feature_cols = ["impressions_90d", "clicks_prev_30d", "pos_clean", "content_age_days", "ctr"]

X_train, y_train = train_df[feature_cols], train_df["target_binary"]
X_test, y_test = test_df[feature_cols], test_df["target_binary"]

# Train Random Forest Classifier
rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
rf.fit(X_train, y_train)

test_probs = rf.predict_proba(X_test)[:, 1]
test_df["model_score"] = test_probs

# Precision@K on Holdout Clients
ranked_test = test_df.sort_values(by="model_score", ascending=False)
prec_50 = y_test.loc[ranked_test.index[:50]].mean()
prec_100 = y_test.loc[ranked_test.index[:100]].mean()
auc_score = roc_auc_score(y_test, test_probs)

print("=== CAPSTONE MODEL RESULTS (CLIENT HOLDOUT TEST SET) ===")
print(f"Client Holdout Test Rows: {len(test_df):,} rows across {test_df['client_id'].nunique()} clients")
print(f"Holdout Test ROC-AUC: {auc_score:.4f}")
print(f"Holdout Precision@50: {prec_50:.4f} ({prec_50:.1%})")
print(f"Holdout Precision@100: {prec_100:.4f} ({prec_100:.1%})")


=== CAPSTONE MODEL RESULTS (CLIENT HOLDOUT TEST SET) ===
Client Holdout Test Rows: 7,611 rows across 7 clients
Holdout Test ROC-AUC: 0.9387
Holdout Precision@50: 1.0000 (100.0%)
Holdout Precision@100: 1.0000 (100.0%)


## 5. Results & Baseline Comparison

Comparing the ML model against random base rate and hand-written rule baseline.

In [5]:
## Historical model-versus-baseline calculation — withdrawn

The calculations in the preceding cells are preserved only to reproduce the original snapshot experiment. Their metrics are not valid prospective estimates because temporal overlap has not been resolved. Do not publish the printed Precision@K or ROC-AUC values.

=== MODEL VS BASELINE EVALUATION COMPARISON ===
Base Rate (Random Selection): 23.2%
Rule Baseline Precision@50: 100.0%
Machine Learning Model Precision@50: 100.0%
Machine Learning Model ROC-AUC: 0.9387


## 6. Ranked recommendations

Exporting final ranked recommendations with reason codes and action labels:

In [6]:
# Assign Action Labels to Top Recommendations
top_50_recommendations = ranked_test.head(50).copy()
top_50_recommendations["action_label"] = np.where(
    top_50_recommendations["traffic_drop"] > 50,
    "REFRESH_AND_EXPAND_CONTENT",
    "OPTIMIZE_TITLE_AND_METAS"
)

print("=== TOP 10 CAPSTONE RECOMMENDATIONS PLAYBOOK ===")
print(top_50_recommendations[["content_id", "client_id", "model_score", "traffic_drop", "action_label"]].head(10).to_string(index=False))


=== TOP 10 CAPSTONE RECOMMENDATIONS PLAYBOOK ===
          content_id         client_id  model_score  traffic_drop             action_label
content_fd468dc56212 client_2c624232cd     0.973545             1 OPTIMIZE_TITLE_AND_METAS
content_9f787a5d8e6f client_f74efabef1     0.965017             1 OPTIMIZE_TITLE_AND_METAS
content_d1bc0303bb56 client_6208ef0f77     0.955322             1 OPTIMIZE_TITLE_AND_METAS
content_c2a3a4d575c1 client_f74efabef1     0.955191             1 OPTIMIZE_TITLE_AND_METAS
content_d8c60980fcfd client_f74efabef1     0.951196             1 OPTIMIZE_TITLE_AND_METAS
content_197eb73ac8e7 client_2c624232cd     0.946344             1 OPTIMIZE_TITLE_AND_METAS
content_380b4510fdfb client_f74efabef1     0.944638             1 OPTIMIZE_TITLE_AND_METAS
content_6a21b377d24c client_2c624232cd     0.943331             1 OPTIMIZE_TITLE_AND_METAS
content_bdee2164f576 client_6208ef0f77     0.941905             1 OPTIMIZE_TITLE_AND_METAS
content_9053114a42eb client_d4735e3a26   

## 7. Artifacts the paper embeds

Exporting metrics JSON receipt to `outputs/model_results.json`:

In [7]:
# Export JSON receipt
output_json = {
    "model_name": "RandomForestClassifier",
    "test_roc_auc": round(float(auc_score), 4),
    "precision_at_50": round(float(prec_50), 4),
    "precision_at_100": round(float(prec_100), 4),
    "holdout_clients": int(test_df["client_id"].nunique()),
    "test_rows": len(test_df)
}

os.makedirs("../../outputs", exist_ok=True)
os.makedirs("outputs", exist_ok=True)
with open("outputs/model_results.json", "w") as f:
    json.dump(output_json, f, indent=2)
print("Successfully saved receipt to outputs/model_results.json")


Successfully saved receipt to outputs/model_results.json


## 8. Closing Summary — Validation Work Required

The capstone demonstrates a prototype workflow for data preparation, model experimentation, and decision-support outputs. Its predictive evaluation is not validated.

Before any model-performance claim is restored, rebuild point-in-time features from dated daily records, define a forward label window after the prediction cutoff, apply chronological validation, and regenerate every report and public artifact from one reproducible run. See [the temporal validation remediation plan](../../docs/temporal-validation-remediation.md).

## Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/`
- [x] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [x] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.